# Postgres CDC to Iceberg with a 5-minute SLA that survives inserts, updates, and deletes

You replicated a Postgres table to Iceberg six months ago as a one-shot. The business asked for ongoing replication and they want change events landed in five minutes. Today you build the pipeline that does that. DMS for the source-side capture, Glue ETL for the sink-side MERGE, Iceberg as the lakehouse. By the end of the lab you can answer the interview question about how CDC actually handles a DELETE.

The five deliverables map to five checkpoints:
1. DMS full-load completed; Iceberg sink contains the seed Postgres rows.
2. INSERT propagation: a canary row inserted in Postgres appears in Iceberg within 5 minutes.
3. UPDATE propagation: an existing row's new value reaches Iceberg within 5 minutes.
4. DELETE propagation: a deleted row is absent from Iceberg within 5 minutes (the gotcha most CDC tutorials skip).
5. Iceberg snapshot history shows distinct snapshots for the full-load and each MERGE.

**Time.** About 100 minutes hands-on. RDS provisioning + DMS replication instance provisioning + full-load is the bulk.

**Cost.** $3.00 to $6.00 per session. Highest-cost lab in the track. **DMS bills continuously from launch until deletion, EVEN WHILE TASKS ARE STOPPED.** The mitigation is the cleanup discipline: cleanup deletes the DMS task first, the DMS instance second, RDS third.

**Critical resources.** RDS Postgres db.t4g.micro AND DMS replication instance dms.t3.micro. The setup cell requires explicit "I understand" acknowledgement of DMS's standing meter.

**Multi-session.** 24-hour JWT with a 2.5-hour critical-window cap.

This lab is part of the Labrun Data Engineering job-prep track, Streaming & CDC sub-track.


In [ ]:
!pip install --quiet labrun-checks==0.7.0 boto3 psycopg2-binary


In [ ]:
# Imports and constants.

import atexit
import io
import json
import re
import sys
import time
import uuid
import zipfile
import secrets as _secrets
from getpass import getpass

import boto3
from botocore.exceptions import ClientError

from labrun_checks import CheckpointResult, CleanupResource, check, cleanup, register, run_cleanup

LAB_SLUG = "cdc-postgres-to-iceberg"
LAB_ID = LAB_SLUG
LAB_TAG_KEY = "labrun:lab-slug"
LAB_TAG_VALUE = LAB_SLUG
REGION = "us-east-1"

BUCKET_NAME = None
DATABASE_NAME = f"labrun_{LAB_SLUG.replace('-', '_')}_db"
SINK_TABLE = "orders_iceberg"
STAGING_TABLE = "orders_staging"

RDS_ID = f"labrun-{LAB_SLUG}-rds"
SG_NAME = f"labrun-{LAB_SLUG}-sg"
SUBNET_GROUP = f"labrun-{LAB_SLUG}-subnet"
DMS_SUBNET_GROUP = f"labrun-{LAB_SLUG}-dms-subnet"
DMS_INSTANCE = f"labrun-{LAB_SLUG}-dms"
DMS_SRC = f"labrun-{LAB_SLUG}-src"
DMS_TGT = f"labrun-{LAB_SLUG}-tgt"
DMS_TASK = f"labrun-{LAB_SLUG}-task"
DMS_ROLE = "dms-vpc-role"  # AWS-managed name pattern
GLUE_ROLE = f"labrun-{LAB_SLUG}-glue-role"
SECRET_NAME = f"labrun-{LAB_SLUG}-secret"
MERGE_JOB = f"labrun-{LAB_SLUG}-merge"
SCHEDULE_NAME = f"labrun-{LAB_SLUG}-schedule"
ATHENA_WG = f"labrun-{LAB_SLUG}-wg"

ACCOUNT_ID = None
SG_ID = None
DMS_INSTANCE_ARN = None
DMS_SRC_ARN = None
DMS_TGT_ARN = None
DMS_TASK_ARN = None
SECRET_ARN = None


In [ ]:
# NBVAL_SKIP
# Setup. Hard confirmation about DMS billing.

print("Paste your lab session token:")
SESSION_TOKEN = getpass("Session token: ").strip()
if not SESSION_TOKEN:
    raise SystemExit("Missing session token.")
AWS_ACCESS_KEY_ID = getpass("AWS_ACCESS_KEY_ID: ").strip()
AWS_SECRET_ACCESS_KEY = getpass("AWS_SECRET_ACCESS_KEY: ").strip()
AWS_SESSION_TOKEN = getpass("AWS_SESSION_TOKEN (Enter to skip): ").strip() or None
AWS_CREDENTIALS = {"aws_access_key_id": AWS_ACCESS_KEY_ID, "aws_secret_access_key": AWS_SECRET_ACCESS_KEY, "aws_session_token": AWS_SESSION_TOKEN}

print()
print("DMS bills continuously from launch until deletion, EVEN WHILE TASKS ARE STOPPED.")
print("RDS Postgres db.t4g.micro is free-tier eligible for new accounts (12 months, 750 hrs/month).")
print("Across this lab the combined meter is roughly $0.05/hour outside the free tier.")
print()
print("Type 'I understand' to acknowledge and proceed:")
ack = input("Acknowledgement: ").strip()
if ack != "I understand":
    raise SystemExit("Acknowledgement not provided.")

sts = boto3.client("sts", region_name=REGION, **{k: v for k, v in AWS_CREDENTIALS.items() if v})
try:
    ACCOUNT_ID = sts.get_caller_identity()["Account"]
except ClientError as exc:
    raise SystemExit(f"AWS credentials rejected: {exc}")

BUCKET_NAME = f"labrun-{LAB_SLUG}-{ACCOUNT_ID}"
print(f"Account: {ACCOUNT_ID}; bucket: {BUCKET_NAME}")
register(SESSION_TOKEN, lab_id=LAB_ID, credentials=AWS_CREDENTIALS)
print("Setup complete.")


In [ ]:
# NBVAL_SKIP
# CLEANUP_MANIFEST. Critical-first per RESOURCE-SAFETY-SPEC.md Lab 9
# mitigation: DMS task -> DMS instance -> RDS -> ...

CLEANUP_MANIFEST = [
    CleanupResource(type="dms_task", id="pending", region=REGION, critical=True),
    CleanupResource(type="dms_instance", id="pending", region=REGION, critical=True),
    CleanupResource(type="dms_endpoint", id="pending-src", region=REGION),
    CleanupResource(type="dms_endpoint", id="pending-tgt", region=REGION),
    CleanupResource(type="rds_instance", id=RDS_ID, region=REGION, critical=True,
        cli_delete_command=f"aws rds delete-db-instance --db-instance-identifier {RDS_ID} --skip-final-snapshot --delete-automated-backups --region {REGION}"),
    CleanupResource(type="db_subnet_group", id=SUBNET_GROUP, region=REGION),
    CleanupResource(type="dms_subnet_group", id=DMS_SUBNET_GROUP, region=REGION),
    CleanupResource(type="security_group", id="pending", region=REGION),
    CleanupResource(type="secret", id="pending", region=REGION),
    CleanupResource(type="eventbridge_rule", id=SCHEDULE_NAME, region=REGION),
    CleanupResource(type="glue_job", id=MERGE_JOB, region=REGION,
        cli_delete_command=f"aws glue delete-job --job-name {MERGE_JOB} --region {REGION}"),
    CleanupResource(type="iam_role", id=GLUE_ROLE, region=REGION),
    CleanupResource(type="iceberg_table", id=f"{DATABASE_NAME}.{SINK_TABLE}", region=REGION),
    CleanupResource(type="glue_table", id=f"{DATABASE_NAME}.{STAGING_TABLE}", region=REGION),
    CleanupResource(type="athena_workgroup", id=ATHENA_WG, region=REGION,
        cli_delete_command=f"aws athena delete-work-group --work-group {ATHENA_WG} --recursive-delete-option --region {REGION}"),
    CleanupResource(type="glue_database", id=DATABASE_NAME, region=REGION),
    CleanupResource(type="s3_bucket", id=BUCKET_NAME, region=REGION),
]


def _atexit_cleanup():
    print("[atexit] DMS + RDS bill 24/7 while alive. Run the cleanup cell if it has not run.")


atexit.register(_atexit_cleanup)

tagging = boto3.client("resourcegroupstaggingapi", region_name=REGION, **{k: v for k, v in AWS_CREDENTIALS.items() if v})
try:
    orphan_arns = [r["ResourceARN"] for r in tagging.get_resources(TagFilters=[{"Key": LAB_TAG_KEY, "Values": [LAB_TAG_VALUE]}]).get("ResourceTagMappingList", [])]
except ClientError as exc:
    raise SystemExit(f"Orphan scan failed: {exc}")

if orphan_arns:
    print("Found existing labrun-tagged resources. Type 'resume' or 'fresh'.")
    for arn in orphan_arns:
        print(f"  EXISTING: {arn}")
    choice = input("Choice: ").strip().lower()
    if choice == "resume":
        print("Resuming.")
    elif choice == "fresh":
        raise SystemExit("Run previous session's cleanup cell first.")
    else:
        raise SystemExit("Invalid choice.")
else:
    print("No orphans.")


## Task 1: Stand up Postgres, seed orders, build the DMS pipeline, let the full-load complete

You provision RDS Postgres, seed 1000 rows into an `orders` table, then build the DMS replication instance, source endpoint (Postgres), target endpoint (S3), and replication task (`full-load-and-cdc` migration type). After the task starts, full-load lands the 1000 rows in S3 as Parquet files under `s3://<bucket>/cdc/<schema>/orders/`. The Glue MERGE job MERGEs them into `orders_iceberg`.

RDS provisioning is 5-7 minutes. DMS instance is 7-10 minutes. The endpoints are seconds; the task start is seconds; full-load on 1000 rows is ~2 minutes.


In [ ]:
# NBVAL_SKIP
# Task 1: provision Postgres + DMS + Glue MERGE job + seed.

s3 = boto3.client("s3", region_name=REGION, **{k: v for k, v in AWS_CREDENTIALS.items() if v})
iam = boto3.client("iam", region_name=REGION, **{k: v for k, v in AWS_CREDENTIALS.items() if v})
ec2 = boto3.client("ec2", region_name=REGION, **{k: v for k, v in AWS_CREDENTIALS.items() if v})
rds = boto3.client("rds", region_name=REGION, **{k: v for k, v in AWS_CREDENTIALS.items() if v})
dms = boto3.client("dms", region_name=REGION, **{k: v for k, v in AWS_CREDENTIALS.items() if v})
secretsmanager = boto3.client("secretsmanager", region_name=REGION, **{k: v for k, v in AWS_CREDENTIALS.items() if v})
glue = boto3.client("glue", region_name=REGION, **{k: v for k, v in AWS_CREDENTIALS.items() if v})
athena = boto3.client("athena", region_name=REGION, **{k: v for k, v in AWS_CREDENTIALS.items() if v})
events = boto3.client("events", region_name=REGION, **{k: v for k, v in AWS_CREDENTIALS.items() if v})

try:
    s3.create_bucket(Bucket=BUCKET_NAME)
    s3.put_bucket_tagging(Bucket=BUCKET_NAME, Tagging={"TagSet": [{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}]})
except ClientError as exc:
    if exc.response["Error"]["Code"] != "BucketAlreadyOwnedByYou":
        raise

try:
    glue.create_database(DatabaseInput={"Name": DATABASE_NAME})
except glue.exceptions.AlreadyExistsException:
    pass
try:
    athena.create_work_group(
        Name=ATHENA_WG,
        Configuration={"ResultConfiguration": {"OutputLocation": f"s3://{BUCKET_NAME}/athena/"}, "EnforceWorkGroupConfiguration": True},
        Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
    )
except Exception:
    pass


def run_athena(query, timeout=120):
    qid = athena.start_query_execution(QueryString=query, WorkGroup=ATHENA_WG, QueryExecutionContext={"Database": DATABASE_NAME})["QueryExecutionId"]
    deadline = time.time() + timeout
    while time.time() < deadline:
        s = athena.get_query_execution(QueryExecutionId=qid)["QueryExecution"]["Status"]["State"]
        if s == "SUCCEEDED":
            return qid
        if s in ("FAILED", "CANCELLED"):
            raise RuntimeError(f"Athena {qid} {s}: {athena.get_query_execution(QueryExecutionId=qid)['QueryExecution']['Status'].get('StateChangeReason')}")
        time.sleep(2)
    raise TimeoutError()


# Create Iceberg sink table.
run_athena(
    f"CREATE TABLE IF NOT EXISTS {DATABASE_NAME}.{SINK_TABLE} ("
    f"  order_id bigint, customer_id string, amount_cents bigint, status string"
    f") LOCATION 's3://{BUCKET_NAME}/{SINK_TABLE}/' "
    f"TBLPROPERTIES ('table_type'='ICEBERG', 'format'='parquet')"
)

# Network for RDS + DMS
default_vpc = ec2.describe_vpcs(Filters=[{"Name": "is-default", "Values": ["true"]}])["Vpcs"][0]
vpc_id = default_vpc["VpcId"]
subnets = ec2.describe_subnets(Filters=[{"Name": "vpc-id", "Values": [vpc_id]}])["Subnets"]
subnet_ids = [s["SubnetId"] for s in subnets[:2]]

import urllib.request
my_ip = urllib.request.urlopen("https://checkip.amazonaws.com").read().decode().strip()

try:
    sg_resp = ec2.create_security_group(GroupName=SG_NAME, Description="labrun cdc", VpcId=vpc_id, TagSpecifications=[{"ResourceType": "security-group", "Tags": [{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}]}])
    SG_ID = sg_resp["GroupId"]
    # Allow caller IP + intra-SG access for DMS replication instance.
    ec2.authorize_security_group_ingress(GroupId=SG_ID, IpPermissions=[
        {"IpProtocol": "tcp", "FromPort": 5432, "ToPort": 5432, "IpRanges": [{"CidrIp": f"{my_ip}/32"}]},
        {"IpProtocol": "tcp", "FromPort": 5432, "ToPort": 5432, "UserIdGroupPairs": [{"GroupId": SG_ID}]},
    ])
except ClientError as exc:
    if exc.response["Error"]["Code"] == "InvalidGroup.Duplicate":
        sgs = ec2.describe_security_groups(Filters=[{"Name": "group-name", "Values": [SG_NAME]}, {"Name": "vpc-id", "Values": [vpc_id]}])["SecurityGroups"]
        SG_ID = sgs[0]["GroupId"]
    else:
        raise

for r in CLEANUP_MANIFEST:
    if r.type == "security_group":
        r.id = SG_ID
        r.cli_delete_command = f"aws ec2 delete-security-group --group-id {SG_ID} --region {REGION}"
        break

try:
    rds.create_db_subnet_group(DBSubnetGroupName=SUBNET_GROUP, DBSubnetGroupDescription="labrun", SubnetIds=subnet_ids, Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}])
except rds.exceptions.DBSubnetGroupAlreadyExistsFault:
    pass

db_password = _secrets.token_urlsafe(16)

try:
    rds.create_db_instance(
        DBInstanceIdentifier=RDS_ID, DBInstanceClass="db.t4g.micro", Engine="postgres",
        AllocatedStorage=20, MasterUsername="labrun", MasterUserPassword=db_password,
        VpcSecurityGroupIds=[SG_ID], DBSubnetGroupName=SUBNET_GROUP,
        PubliclyAccessible=True, BackupRetentionPeriod=0,
        Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
    )
    print("  RDS Postgres creating; hold for ~6 minutes...")
except rds.exceptions.DBInstanceAlreadyExistsFault:
    pass

rds.get_waiter("db_instance_available").wait(DBInstanceIdentifier=RDS_ID, WaiterConfig={"Delay": 30, "MaxAttempts": 30})
ep = rds.describe_db_instances(DBInstanceIdentifier=RDS_ID)["DBInstances"][0]["Endpoint"]
print(f"  RDS available at {ep['Address']}:{ep['Port']}")

# Store secret
try:
    secret_resp = secretsmanager.create_secret(
        Name=SECRET_NAME,
        SecretString=json.dumps({"username": "labrun", "password": db_password, "host": ep["Address"], "port": ep["Port"], "database": "postgres"}),
        Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
    )
    SECRET_ARN = secret_resp["ARN"]
except secretsmanager.exceptions.ResourceExistsException:
    SECRET_ARN = secretsmanager.describe_secret(SecretId=SECRET_NAME)["ARN"]
    secretsmanager.put_secret_value(SecretId=SECRET_NAME, SecretString=json.dumps({"username": "labrun", "password": db_password, "host": ep["Address"], "port": ep["Port"], "database": "postgres"}))

for r in CLEANUP_MANIFEST:
    if r.type == "secret":
        r.id = SECRET_ARN
        r.cli_delete_command = f"aws secretsmanager delete-secret --secret-id {SECRET_NAME} --force-delete-without-recovery --region {REGION}"
        break

# Seed orders + REPLICA IDENTITY FULL (required for DMS DELETE capture)
import psycopg2
conn = psycopg2.connect(host=ep["Address"], port=ep["Port"], dbname="postgres", user="labrun", password=db_password)
conn.autocommit = True
with conn.cursor() as cur:
    cur.execute("CREATE TABLE IF NOT EXISTS orders (order_id bigint primary key, customer_id text, amount_cents bigint, status text)")
    cur.execute("ALTER TABLE orders REPLICA IDENTITY FULL")
    cur.execute("TRUNCATE orders")
    rows = [(i, f"cust-{i % 100}", (i * 13) % 50000, "paid") for i in range(1, 1001)]
    cur.executemany("INSERT INTO orders (order_id, customer_id, amount_cents, status) VALUES (%s, %s, %s, %s)", rows)
conn.close()
print("  orders table seeded with 1000 rows")

# DMS subnet group
try:
    dms.create_replication_subnet_group(
        ReplicationSubnetGroupIdentifier=DMS_SUBNET_GROUP,
        ReplicationSubnetGroupDescription="labrun",
        SubnetIds=subnet_ids,
        Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
    )
except dms.exceptions.ResourceAlreadyExistsFault:
    pass

# DMS replication instance
try:
    dms_inst = dms.create_replication_instance(
        ReplicationInstanceIdentifier=DMS_INSTANCE,
        ReplicationInstanceClass="dms.t3.micro",
        AllocatedStorage=20,
        VpcSecurityGroupIds=[SG_ID],
        ReplicationSubnetGroupIdentifier=DMS_SUBNET_GROUP,
        PubliclyAccessible=True,
        Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
    )
    DMS_INSTANCE_ARN = dms_inst["ReplicationInstance"]["ReplicationInstanceArn"]
    print("  DMS replication instance creating; hold for ~8 minutes...")
except dms.exceptions.ResourceAlreadyExistsFault:
    inst_desc = dms.describe_replication_instances(Filters=[{"Name": "replication-instance-id", "Values": [DMS_INSTANCE]}])["ReplicationInstances"][0]
    DMS_INSTANCE_ARN = inst_desc["ReplicationInstanceArn"]

# Wait until available
deadline = time.time() + 900
while time.time() < deadline:
    inst = dms.describe_replication_instances(Filters=[{"Name": "replication-instance-id", "Values": [DMS_INSTANCE]}])["ReplicationInstances"][0]
    if inst["ReplicationInstanceStatus"] == "available":
        break
    time.sleep(30)
else:
    raise RuntimeError("DMS instance did not become available within 15 min")

for r in CLEANUP_MANIFEST:
    if r.type == "dms_instance":
        r.id = DMS_INSTANCE_ARN
        r.cli_delete_command = f"aws dms delete-replication-instance --replication-instance-arn {DMS_INSTANCE_ARN} --region {REGION}"
        break

print(f"  DMS instance available: {DMS_INSTANCE_ARN}")

# DMS Glue IAM role for the MERGE job
glue_trust = {"Version": "2012-10-17", "Statement": [{"Effect": "Allow", "Principal": {"Service": "glue.amazonaws.com"}, "Action": "sts:AssumeRole"}]}
glue_inline = {"Version": "2012-10-17", "Statement": [{"Effect": "Allow", "Action": ["s3:*", "athena:*", "glue:*", "logs:*"], "Resource": "*"}]}
try:
    iam.create_role(RoleName=GLUE_ROLE, AssumeRolePolicyDocument=json.dumps(glue_trust), Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}])
    iam.attach_role_policy(RoleName=GLUE_ROLE, PolicyArn="arn:aws:iam::aws:policy/service-role/AWSGlueServiceRole")
    iam.put_role_policy(RoleName=GLUE_ROLE, PolicyName="labrun-glue-inline", PolicyDocument=json.dumps(glue_inline))
except iam.exceptions.EntityAlreadyExistsException:
    pass
print("  Glue IAM propagation, hold 15 seconds...")
time.sleep(15)

# YOUR CODE: Create the DMS source endpoint (Postgres) and target endpoint (S3).
# YOUR CODE: Create the DMS task (full-load-and-cdc).
# YOUR CODE: Start the task and poll until full-load = 100%.

# Upload the MERGE script.
MERGE_SCRIPT = '''
import sys
from awsglue.context import GlueContext
from awsglue.job import Job
from awsglue.utils import getResolvedOptions
from pyspark.context import SparkContext
from pyspark.sql.functions import col

args = getResolvedOptions(sys.argv, ["JOB_NAME", "DATABASE_NAME", "SINK_TABLE", "STAGING_TABLE", "BUCKET"])
sc = SparkContext.getOrCreate()
glue_ctx = GlueContext(sc)
spark = glue_ctx.spark_session
job = Job(glue_ctx)
job.init(args["JOB_NAME"], args)

# Read DMS CDC output (Parquet under s3://<bucket>/cdc/.../).
cdc_path = "s3://" + args["BUCKET"] + "/cdc/"
try:
    cdc = spark.read.parquet(cdc_path + "*/*/*.parquet")
except Exception:
    print("LABRUN_NO_CDC_FILES_YET")
    job.commit()
    sys.exit(0)

# Latest record per order_id (op + most recent timestamp).
# DMS emits "Op" column: I/U/D for inserts/updates/deletes.
cdc.createOrReplaceTempView("cdc_raw")

# Deduplicate: keep the last op per order_id within this batch.
dedup_sql = (
    "SELECT order_id, customer_id, amount_cents, status, Op FROM ("
    "  SELECT *, ROW_NUMBER() OVER (PARTITION BY order_id ORDER BY input_file_name() DESC) AS rn "
    "  FROM cdc_raw"
    ") WHERE rn = 1"
)
deduped = spark.sql(dedup_sql)
deduped.createOrReplaceTempView("cdc_delta")

# MERGE INTO with DELETE branch.
merge_sql = (
    "MERGE INTO glue_catalog." + args["DATABASE_NAME"] + "." + args["SINK_TABLE"] + " t "
    "USING cdc_delta s ON t.order_id = s.order_id "
    "WHEN MATCHED AND s.Op = 'D' THEN DELETE "
    "WHEN MATCHED AND s.Op IN ('U', 'I') THEN UPDATE SET customer_id = s.customer_id, amount_cents = s.amount_cents, status = s.status "
    "WHEN NOT MATCHED AND s.Op IN ('I', 'U') THEN INSERT (order_id, customer_id, amount_cents, status) "
    "                                                VALUES (s.order_id, s.customer_id, s.amount_cents, s.status)"
)
spark.sql(merge_sql)
print("LABRUN_MERGE_OK")
job.commit()
'''.lstrip()

s3.put_object(Bucket=BUCKET_NAME, Key="scripts/merge.py", Body=MERGE_SCRIPT.encode("utf-8"))

iceberg_conf = (
    "spark.sql.catalog.glue_catalog=org.apache.iceberg.spark.SparkCatalog "
    f"--conf spark.sql.catalog.glue_catalog.warehouse=s3://{BUCKET_NAME}/warehouse/ "
    "--conf spark.sql.catalog.glue_catalog.catalog-impl=org.apache.iceberg.aws.glue.GlueCatalog "
    "--conf spark.sql.catalog.glue_catalog.io-impl=org.apache.iceberg.aws.s3.S3FileIO "
    "--conf spark.sql.extensions=org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions"
)
glue_role_arn = f"arn:aws:iam::{ACCOUNT_ID}:role/{GLUE_ROLE}"

try:
    glue.create_job(
        Name=MERGE_JOB, Role=glue_role_arn,
        Command={"Name": "glueetl", "ScriptLocation": f"s3://{BUCKET_NAME}/scripts/merge.py", "PythonVersion": "3"},
        DefaultArguments={"--DATABASE_NAME": DATABASE_NAME, "--SINK_TABLE": SINK_TABLE, "--STAGING_TABLE": STAGING_TABLE,
                          "--BUCKET": BUCKET_NAME, "--datalake-formats": "iceberg", "--conf": iceberg_conf,
                          "--TempDir": f"s3://{BUCKET_NAME}/temp/"},
        GlueVersion="4.0", WorkerType="G.1X", NumberOfWorkers=2, Timeout=15,
        Tags={LAB_TAG_KEY: LAB_TAG_VALUE},
    )
except glue.exceptions.AlreadyExistsException:
    pass
print(f"  Glue MERGE job ready: {MERGE_JOB}")


In [ ]:
# NBVAL_SKIP
# OBSERVE cell: poll DMS task progress every 10s for up to 10 min.

def observe_dms_task_phase():
    if not DMS_TASK_ARN:
        print("DMS task not yet created. Run YOUR CODE block above to create + start the task.")
        return
    deadline = time.time() + 600
    while time.time() < deadline:
        task = dms.describe_replication_tasks(Filters=[{"Name": "replication-task-arn", "Values": [DMS_TASK_ARN]}])["ReplicationTasks"][0]
        stats = task.get("ReplicationTaskStats", {})
        full_load_pct = stats.get("FullLoadProgressPercent", 0)
        status = task.get("Status")
        print(f"  status={status} full_load_pct={full_load_pct}")
        if full_load_pct == 100 or status == "stopped" or status == "running":
            if full_load_pct == 100:
                return
        time.sleep(10)


# YOUR CODE: call observe_dms_task_phase() to watch full-load complete.
# Then trigger the first Glue MERGE job to land the rows into Iceberg.


In [ ]:
# NBVAL_SKIP
# Checkpoint 1: full-load == 100% + orders_iceberg has 1000 rows.

def validator_1(session):
    if not DMS_TASK_ARN:
        return CheckpointResult("fail", error_reason="DMS_TASK_ARN not set. Create + start the DMS task.")
    try:
        task = dms.describe_replication_tasks(Filters=[{"Name": "replication-task-arn", "Values": [DMS_TASK_ARN]}])["ReplicationTasks"][0]
    except ClientError as exc:
        return CheckpointResult("error", error_reason=f"DMS describe failed: {exc}")
    stats = task.get("ReplicationTaskStats", {})
    if stats.get("FullLoadProgressPercent", 0) < 100:
        return CheckpointResult("fail", error_reason=f"FullLoadProgressPercent is {stats.get('FullLoadProgressPercent')}; need 100.")
    # Check Iceberg row count
    try:
        qid = run_athena(f"SELECT count(*) FROM {DATABASE_NAME}.{SINK_TABLE}")
        count = int(athena.get_query_results(QueryExecutionId=qid)["ResultSet"]["Rows"][1]["Data"][0]["VarCharValue"])
    except Exception as exc:
        return CheckpointResult("error", error_reason=f"Athena count failed: {exc}")
    if count != 1000:
        return CheckpointResult("fail", error_reason=f"orders_iceberg has {count} rows; expected 1000. Did the MERGE job run after full-load completed?")
    return CheckpointResult("pass")


check(1, validator_1)


<details><summary>Hint 1 (nudge)</summary>

DMS needs a source endpoint (Postgres), a target endpoint (S3), and a task that ties them together with `MigrationType="full-load-and-cdc"`.

</details>

<details><summary>Hint 2 (direction)</summary>

Source endpoint: `EngineName="postgres"`, point at the RDS host. Target endpoint: `EngineName="s3"`, `S3Settings={"BucketName": ..., "BucketFolder": "cdc/", "DataFormat": "parquet", "IncludeOpForFullLoad": True, "CdcInsertsAndUpdates": True}`. Task: `TableMappings={"rules": [{"rule-type": "selection", "rule-id": "1", "rule-name": "1", "object-locator": {"schema-name": "public", "table-name": "orders"}, "rule-action": "include"}]}`.

</details>

<details><summary>Hint 3 (spoiler)</summary>

```python
src = dms.create_endpoint(EndpointIdentifier=DMS_SRC, EndpointType="source", EngineName="postgres",
    Username="labrun", Password=db_password, ServerName=ep["Address"], Port=ep["Port"], DatabaseName="postgres",
    Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}])
DMS_SRC_ARN = src["Endpoint"]["EndpointArn"]

tgt = dms.create_endpoint(EndpointIdentifier=DMS_TGT, EndpointType="target", EngineName="s3",
    S3Settings={
        "BucketName": BUCKET_NAME, "BucketFolder": "cdc/",
        "DataFormat": "parquet", "IncludeOpForFullLoad": True, "CdcInsertsAndUpdates": True,
        "ServiceAccessRoleArn": f"arn:aws:iam::{ACCOUNT_ID}:role/dms-vpc-role",  # AWS-managed standard
    },
    Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}])
DMS_TGT_ARN = tgt["Endpoint"]["EndpointArn"]

for r in CLEANUP_MANIFEST:
    if r.type == "dms_endpoint" and "src" in r.id:
        r.id = DMS_SRC_ARN
    if r.type == "dms_endpoint" and "tgt" in r.id:
        r.id = DMS_TGT_ARN

task = dms.create_replication_task(
    ReplicationTaskIdentifier=DMS_TASK,
    SourceEndpointArn=DMS_SRC_ARN, TargetEndpointArn=DMS_TGT_ARN,
    ReplicationInstanceArn=DMS_INSTANCE_ARN,
    MigrationType="full-load-and-cdc",
    TableMappings=json.dumps({"rules": [{"rule-type": "selection", "rule-id": "1", "rule-name": "1",
                                          "object-locator": {"schema-name": "public", "table-name": "orders"},
                                          "rule-action": "include"}]}),
    Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
)
DMS_TASK_ARN = task["ReplicationTask"]["ReplicationTaskArn"]

for r in CLEANUP_MANIFEST:
    if r.type == "dms_task":
        r.id = DMS_TASK_ARN
        break

dms.start_replication_task(ReplicationTaskArn=DMS_TASK_ARN, StartReplicationTaskType="start-replication")
observe_dms_task_phase()

# Trigger MERGE
glue.start_job_run(JobName=MERGE_JOB)
```

</details>


**Wallet check.** RDS + DMS instance both alive: combined meter ~$0.035/hour. First Glue MERGE run at 10-min min: ~$0.15. Across the lab the steady spend dominates. Session total so far: ~25 cents.

## Task 2: INSERT propagation. Canary row in Postgres within 5 minutes lands in Iceberg.

Insert a canary row (`order_id = -1`) in Postgres. Run the Glue MERGE job to land the change. Poll Iceberg until the row appears.


In [ ]:
# NBVAL_SKIP
# Task 2: INSERT propagation.

# YOUR CODE: psycopg2 INSERT order_id=-1 customer='lab16-canary' amount=999 status='paid'.
# YOUR CODE: trigger MERGE; poll for the row in Iceberg.


In [ ]:
# NBVAL_SKIP
# Checkpoint 2: order_id=-1 appears in orders_iceberg.

def validator_2(session):
    deadline = time.time() + 360
    while time.time() < deadline:
        try:
            qid = run_athena(f"SELECT count(*) FROM {DATABASE_NAME}.{SINK_TABLE} WHERE order_id = -1")
            count = int(athena.get_query_results(QueryExecutionId=qid)["ResultSet"]["Rows"][1]["Data"][0]["VarCharValue"])
            if count == 1:
                return CheckpointResult("pass")
        except Exception:
            pass
        time.sleep(20)
    return CheckpointResult("fail", error_reason="Canary row did not appear in orders_iceberg within 6 minutes.")


check(2, validator_2)


<details><summary>Hint 1 (nudge)</summary>

DMS captures the INSERT on commit. Your MERGE job needs to fire to land the change in Iceberg.

</details>

<details><summary>Hint 2 (direction)</summary>

`psycopg2.connect(...)` + `INSERT INTO orders ...` + commit. Then `glue.start_job_run(JobName=MERGE_JOB)`. Wait ~2 minutes for the MERGE to complete; the validator polls for up to 6 minutes.

</details>

<details><summary>Hint 3 (spoiler)</summary>

```python
import psycopg2
secret = json.loads(secretsmanager.get_secret_value(SecretId=SECRET_NAME)["SecretString"])
conn = psycopg2.connect(host=secret["host"], port=secret["port"], dbname=secret["database"], user=secret["username"], password=secret["password"])
conn.autocommit = True
with conn.cursor() as cur:
    cur.execute("INSERT INTO orders (order_id, customer_id, amount_cents, status) VALUES (-1, 'lab16-canary', 999, 'paid')")
conn.close()
time.sleep(60)  # let DMS replicate
glue.start_job_run(JobName=MERGE_JOB)
```

</details>


**Wallet check.** Insert + MERGE run: ~$0.15. Session total: ~40 cents.

## Task 3: UPDATE propagation. New amount_cents reaches Iceberg.

Update the canary row's amount_cents to 7777. Run MERGE. Confirm.


In [ ]:
# NBVAL_SKIP
# Task 3: UPDATE propagation.

# YOUR CODE: psycopg2 UPDATE orders SET amount_cents = 7777 WHERE order_id = -1.
# YOUR CODE: trigger MERGE; poll for the new value.


In [ ]:
# NBVAL_SKIP
# Checkpoint 3: order_id=-1 has amount_cents = 7777.

def validator_3(session):
    deadline = time.time() + 360
    while time.time() < deadline:
        try:
            qid = run_athena(f"SELECT amount_cents FROM {DATABASE_NAME}.{SINK_TABLE} WHERE order_id = -1")
            rows = athena.get_query_results(QueryExecutionId=qid)["ResultSet"]["Rows"]
            if len(rows) > 1:
                val = int(rows[1]["Data"][0]["VarCharValue"])
                if val == 7777:
                    return CheckpointResult("pass")
        except Exception:
            pass
        time.sleep(20)
    return CheckpointResult("fail", error_reason="UPDATE did not propagate to orders_iceberg within 6 minutes.")


check(3, validator_3)


<details><summary>Hint 1 (nudge)</summary>

The MERGE script's `WHEN MATCHED AND s.Op IN ('U', 'I') THEN UPDATE` branch handles updates.

</details>

<details><summary>Hint 2 (direction)</summary>

Run the UPDATE; wait ~60s for DMS to capture it; trigger MERGE; poll Iceberg.

</details>

<details><summary>Hint 3 (spoiler)</summary>

```python
with conn.cursor() as cur:
    cur.execute("UPDATE orders SET amount_cents = 7777 WHERE order_id = -1")
time.sleep(60)
glue.start_job_run(JobName=MERGE_JOB)
```

</details>


**Wallet check.** MERGE run #3: ~$0.15. Session total: ~55 cents.

## Task 4: DELETE propagation. Deleted row is absent from Iceberg.

The most-skipped CDC gotcha. DMS emits `Op = 'D'` on deletes; the MERGE script's `WHEN MATCHED AND s.Op = 'D' THEN DELETE` clause handles it. Without that clause, deletes silently fail and the sink stays out of sync with the source.

Verify the MERGE script has the DELETE clause. Delete the canary row. Trigger MERGE. Confirm absence.


In [ ]:
# NBVAL_SKIP
# Task 4: DELETE propagation.

# YOUR CODE: verify the MERGE script contains
#   "WHEN MATCHED AND s.Op = 'D' THEN DELETE"
# (the lab provides this script; if you re-wrote it without that clause,
#  the validator will catch the issue.)

# YOUR CODE: psycopg2 DELETE FROM orders WHERE order_id = -1.
# YOUR CODE: trigger MERGE; poll for the row to disappear.


In [ ]:
# NBVAL_SKIP
# Checkpoint 4: order_id=-1 absent from orders_iceberg.

def validator_4(session):
    deadline = time.time() + 360
    while time.time() < deadline:
        try:
            qid = run_athena(f"SELECT count(*) FROM {DATABASE_NAME}.{SINK_TABLE} WHERE order_id = -1")
            count = int(athena.get_query_results(QueryExecutionId=qid)["ResultSet"]["Rows"][1]["Data"][0]["VarCharValue"])
            if count == 0:
                return CheckpointResult("pass")
        except Exception:
            pass
        time.sleep(20)
    return CheckpointResult("fail", error_reason="DELETE did not propagate to orders_iceberg within 6 minutes. Check the MERGE script has WHEN MATCHED AND s.Op = 'D' THEN DELETE.")


check(4, validator_4)


<details><summary>Hint 1 (nudge)</summary>

Most CDC tutorials skip the DELETE branch. Yours does not.

</details>

<details><summary>Hint 2 (direction)</summary>

The MERGE script's three branches: `WHEN MATCHED AND s.Op = 'D' THEN DELETE`, `WHEN MATCHED AND s.Op IN ('U', 'I') THEN UPDATE`, `WHEN NOT MATCHED AND s.Op IN ('I', 'U') THEN INSERT`.

</details>

<details><summary>Hint 3 (spoiler)</summary>

The MERGE script is provided fully wired (uploaded during Task 1 setup). Run:

```python
with conn.cursor() as cur:
    cur.execute("DELETE FROM orders WHERE order_id = -1")
time.sleep(60)
glue.start_job_run(JobName=MERGE_JOB)
```

</details>


**Wallet check.** MERGE run #4: ~$0.15. Session total: ~70 cents.

## Task 5: Iceberg snapshot history shows distinct snapshots.

Each MERGE run that mutates the table creates a snapshot. After full-load + INSERT + UPDATE + DELETE you should see at least 4 snapshots.


In [ ]:
# NBVAL_SKIP
# Task 5: snapshot history.

# YOUR CODE: SELECT count(*) FROM "<db>"."<table>$snapshots".
# Assign to SNAPSHOT_COUNT.


In [ ]:
# NBVAL_SKIP
# Checkpoint 5: snapshot count >= 4.

def validator_5(session):
    n = globals().get("SNAPSHOT_COUNT")
    if n is None:
        return CheckpointResult("fail", error_reason="SNAPSHOT_COUNT not set.")
    if n < 4:
        return CheckpointResult("fail", error_reason=f"Iceberg has {n} snapshots; expected at least 4 (full-load + 3 MERGE runs).")
    return CheckpointResult("pass")


check(5, validator_5)


<details><summary>Hint 1 (nudge)</summary>

Iceberg metadata tables: `<table>$snapshots`, `<table>$history`, `<table>$files`. Athena supports all of them.

</details>

<details><summary>Hint 2 (direction)</summary>

```sql
SELECT count(*) FROM "<db>"."orders_iceberg$snapshots"
```

</details>

<details><summary>Hint 3 (spoiler)</summary>

```python
qid = run_athena(f'SELECT count(*) FROM "{DATABASE_NAME}"."{SINK_TABLE}$snapshots"')
SNAPSHOT_COUNT = int(athena.get_query_results(QueryExecutionId=qid)["ResultSet"]["Rows"][1]["Data"][0]["VarCharValue"])
print(SNAPSHOT_COUNT)
```

</details>


**Wallet check.** Tiny Athena scan on metadata table: nearly free. Session total: ~70 cents.

In [ ]:
# NBVAL_SKIP
# Cleanup. DMS task -> DMS instance -> RDS -> ... in order.

from labrun_checks.adapters.aws import AwsCleanupAdapter


class _LabAdapter(AwsCleanupAdapter):
    def delete_resource(self, credentials, resource):
        if resource.type == "dms_task":
            client = boto3.client("dms", region_name=resource.region, **{k: v for k, v in credentials.items() if v})
            if resource.id == "pending":
                return
            try:
                client.stop_replication_task(ReplicationTaskArn=resource.id)
                # wait until stopped
                deadline = time.time() + 600
                while time.time() < deadline:
                    s = client.describe_replication_tasks(Filters=[{"Name": "replication-task-arn", "Values": [resource.id]}])["ReplicationTasks"][0]["Status"]
                    if s in ("stopped", "failed", "ready"):
                        break
                    time.sleep(15)
            except client.exceptions.InvalidResourceStateFault:
                pass
            except ClientError:
                pass
            try:
                client.delete_replication_task(ReplicationTaskArn=resource.id)
            except ClientError:
                pass
            return
        if resource.type == "dms_instance":
            client = boto3.client("dms", region_name=resource.region, **{k: v for k, v in credentials.items() if v})
            if resource.id == "pending":
                return
            try:
                client.delete_replication_instance(ReplicationInstanceArn=resource.id)
            except ClientError:
                pass
            return
        if resource.type == "dms_endpoint":
            client = boto3.client("dms", region_name=resource.region, **{k: v for k, v in credentials.items() if v})
            if resource.id.startswith("pending"):
                return
            try:
                client.delete_endpoint(EndpointArn=resource.id)
            except ClientError:
                pass
            return
        if resource.type == "dms_subnet_group":
            client = boto3.client("dms", region_name=resource.region, **{k: v for k, v in credentials.items() if v})
            try:
                client.delete_replication_subnet_group(ReplicationSubnetGroupIdentifier=resource.id)
            except ClientError:
                pass
            return
        if resource.type == "iceberg_table":
            db, table = resource.id.split(".", 1)
            client = boto3.client("glue", region_name=resource.region, **{k: v for k, v in credentials.items() if v})
            try:
                client.delete_table(DatabaseName=db, Name=table)
            except client.exceptions.EntityNotFoundException:
                pass
            return
        if resource.type == "glue_table":
            db, table = resource.id.split(".", 1)
            client = boto3.client("glue", region_name=resource.region, **{k: v for k, v in credentials.items() if v})
            try:
                client.delete_table(DatabaseName=db, Name=table)
            except client.exceptions.EntityNotFoundException:
                pass
            return
        if resource.type == "athena_workgroup":
            client = boto3.client("athena", region_name=resource.region, **{k: v for k, v in credentials.items() if v})
            try:
                client.delete_work_group(WorkGroup=resource.id, RecursiveDeleteOption=True)
            except Exception:
                pass
            return
        if resource.type == "secret":
            client = boto3.client("secretsmanager", region_name=resource.region, **{k: v for k, v in credentials.items() if v})
            if resource.id == "pending":
                return
            try:
                client.delete_secret(SecretId=resource.id, ForceDeleteWithoutRecovery=True)
            except Exception:
                pass
            return
        return super().delete_resource(credentials, resource)

    def describe_resource(self, credentials, resource):
        if resource.type in ("dms_task", "dms_instance", "dms_endpoint", "dms_subnet_group",
                              "iceberg_table", "glue_table", "athena_workgroup", "secret"):
            return False
        return super().describe_resource(credentials, resource)


print("Cleanup starts with DMS task (stop + delete, ~2 min), then DMS instance (~3 min), then RDS (~3 min).")
result = run_cleanup(CLEANUP_MANIFEST, adapter=_LabAdapter())

print()
print("Cleanup complete.")
critical_count = sum(1 for r in CLEANUP_MANIFEST if r.critical)
standard_count = len(CLEANUP_MANIFEST) - critical_count
print(f"Critical resources destroyed: {critical_count}")
print(f"Standard resources destroyed: {standard_count}")
failed = len(result.warnings) + len(result.orphans)
print(f"Resources that failed to delete: {failed}")
if failed > 0:
    print("If failed > 0, your AWS account may still incur charges. Resolve before closing this notebook.")
    for w in result.warnings:
        print(f"  WARNING: {w}")
    for o in result.orphans:
        print(f"  ORPHAN: {o}")

cleanup(status=result.status)
if result.status == "dirty":
    sys.exit(1)


**Session total: $3.00 to $6.00.** The two hourly meters (DMS + RDS) dominate. Glue MERGE runs (4-6) add about $1. Cleanup deletes DMS task first because the instance cannot delete while the task is running.

## These are not graded. They are for you.

**1. The DELETE branch.** Walk through what happens when DMS emits `Op = D` and your MERGE does not have the DELETE clause. The MERGE silently does nothing on those rows; the sink stays out of sync. What would you change if you wanted to retain deleted rows for audit (soft-delete pattern)?

**2. Snapshot growth.** At two MERGE runs per minute (one per 5-min schedule plus extras), Iceberg accumulates ~2880 snapshots per day. Metadata reads slow to a crawl after a few weeks. What is the right move?

## Exam-Style Practice

**Question 1.** Your CDC pipeline has been running for two weeks. Iceberg snapshot count exceeds 4000 and metadata reads are slow. What is the right move?

A) Vacuum daily snapshots older than 7 days.

B) Add more DMS shards to spread the write load.

C) Repartition the sink to reduce metadata volume.

D) Switch from Iceberg to Hive.

<details><summary>Show answer</summary>

**A**.

- A) Right. `expire_snapshots(older_than => ...)` is the canonical Iceberg ops hygiene. Two weeks of 5-minute MERGE runs produce 4000 snapshots; expiring at 7 days keeps the metadata tractable while preserving recent rollback capability.
- B) Wrong. DMS does not have "shards". Replication-instance sizing is the lever for throughput, not metadata growth.
- C) Wrong. Repartitioning data does not reduce snapshot count.
- D) Wrong. Hive has its own metadata performance problems. Switching formats is a major change with its own costs; it is not the right response to "too many snapshots".

</details>

**Question 2.** Your CDC MERGE handles `Op = I` and `Op = U` but you skipped `Op = D`. Two weeks in, the source table has had 50,000 deletes; your sink reports a row count 50,000 too high. The smallest fix is:

A) Add the `WHEN MATCHED AND s.Op = 'D' THEN DELETE` clause and reprocess the CDC archive.

B) Rebuild the sink with full-load-and-cdc from scratch.

C) Switch from MERGE to OVERWRITE on every run.

D) Add a flag column `deleted_at` and never actually delete rows (soft-delete pattern).

<details><summary>Show answer</summary>

**A**.

- A) Right. Add the missing clause; reprocess the CDC files DMS has on S3 (or the retained CDC history from the source); the MERGE handles deletes going forward AND catches up.
- B) Wrong. Full reprovisioning is expensive (hours, depending on source size) and unnecessary; the CDC log has what you need.
- C) Wrong. OVERWRITE on every run defeats CDC's purpose; you would be re-reading the entire source each schedule, not just changes.
- D) Wrong. Soft-delete is a different requirement, not a fix for the missing DELETE branch. It also requires the downstream consumers to filter on `deleted_at IS NULL`, which is a contract change.

</details>
